### The Core Protocol — Two Methods
```shell
Iterable   → has __iter__()           returns an iterator
Iterator   → has __iter__() + __next__()   returns self + next value
```
> **Every iterator is an iterable**.

> **Not every iterable is an iterator**.

In [ ]:
# The entire protocol in its simplest form:

class MyIterator:
    def __iter__(self):   # makes it an Iterable  → can be used in for loop
        return self       # returns itself         → also makes it an Iterator

    def __next__(self):   # makes it an Iterator  → can call next() on it
        ...               # return next value OR raise StopIteration

**How a for Loop Actually Works**

This is what Python does under the hood — **every single time**

In [ ]:
for item in collection:
    print(item)

# Python translates this to EXACTLY:

iterator = iter(collection)       # calls collection.__iter__()
while True:
    try:
        item = next(iterator)     # calls iterator.__next__()
        print(item)
    except StopIteration:         # __next__ signals "I'm done"
        break                     # loop ends cleanly

**Give `next(iterator, default)` a default value to return instead of raising `StopIteration`**

In [ ]:
obj = range(4)
it_obj = iter(obj)

while True:
    item = next(it_obj, None)  # give None as default value
    if item is None:
        break
    print(item)

0
1
2
3


In [3]:
# You can prove this manually:
numbers = [10, 20, 30]

iterator = iter(numbers)          # list.__iter__() → returns list_iterator object
print(type(iterator))             # <class 'list_iterator'>

print(next(iterator))             # 10
print(next(iterator))             # 20
print(next(iterator))             # 30
print(next(iterator))             # raises StopIteration — list is exhausted

<class 'list_iterator'>
10
20
30


StopIteration: 

**Iterable vs Iterator — The Critical Distinction**

In [ ]:
numbers = [1, 2, 3]   # ITERABLE — has __iter__, no __next__

# Proof:
print(hasattr(numbers, "__iter__"))   # True
print(hasattr(numbers, "__next__"))   # False ← not an iterator!

iterator = iter(numbers)              # NOW we have an iterator
print(hasattr(iterator, "__iter__"))  # True
print(hasattr(iterator, "__next__"))  # True ← iterator!

# KEY difference — an iterable can produce FRESH iterators repeatedly:
for x in numbers: print(x)   # works ✅
for x in numbers: print(x)   # works again ✅ — fresh iterator each time

# An iterator is ONE-WAY, ONE-TIME:
it = iter(numbers)
for x in it: print(x)   # works ✅
for x in it: print(x)   # prints NOTHING — iterator already exhausted!

```shell
                    __iter__    __next__
                   ─────────  ──────────
list, dict, str       ✅          ❌      → Iterable only
list_iterator         ✅          ✅      → Iterator (and Iterable)
your custom class     ✅          ✅      → Iterator (and Iterable)
generator             ✅          ✅      → Iterator (and Iterable)
```

---

**Example 1 - Iterable IS Iterator**

In [ ]:
class SelfIterator:
    def __init__(self, start: int, stop: int, step: int):
        self.start = start
        self.stop = stop
        self.step = step

    def __iter__(self):
        return Iterator(self)


class Iterator:
    def __init__(self, source: SelfIterator):
        self.source = source
        self.start = self.source.start

    def __iter__(self):
        return self
    
    def __next__(self):
        if self.start >= self.source.stop:
            raise StopIteration
        value = self.start
        self.start += self.source.step
        return value

In [4]:
it = SelfIterator(0, 10, 2)

print(list(it))
print(list(it))  # exhausted

[0, 2, 4, 6, 8]
[0, 2, 4, 6, 8]


**Example 2 — Separating Iterable from Iterator (the clean pattern)**

In [15]:
class NumberRange:
    """
    ITERABLE — holds the data, produces fresh iterators on demand.
    Separating iterable from iterator lets you loop over it multiple times.
    """
    def __init__(self, start: int, stop: int, step: int = 1):
        self.start = start
        self.stop  = stop
        self.step  = step

    def __iter__(self):
        return NumberRangeIterator(self)  # ← fresh iterator every time
        #                          ↑ passes self so iterator can read config


class NumberRangeIterator:
    """
    ITERATOR — tracks position, has __next__, exhausts after one pass.
    """
    def __init__(self, source: NumberRange):
        self.source  = source
        self.current = source.start     # ← own position, doesn't mutate source

    def __iter__(self):
        return self                     # iterators must also be iterable

    def __next__(self):
        if self.current >= self.source.stop:
            raise StopIteration
        value        = self.current
        self.current += self.source.step
        return value

In [16]:
# The payoff — reusable iterable:
r = NumberRange(0, 10, 2)

print(list(r))     # [0, 2, 4, 6, 8]
print(list(r))     # [0, 2, 4, 6, 8]  ← still works, fresh iterator!

[0, 2, 4, 6, 8]
[0, 2, 4, 6, 8]


**Example 3 — Countdown Iterator (simplest possible)**

In [10]:
class Countdown:
    """
    Counts down from start to 0.
    Demonstrates the bare minimum iterator implementation.
    """
    def __init__(self, start: int):
        self.current = start

    def __iter__(self):
        return self     # I AM my own iterator

    def __next__(self):
        if self.current < 0:
            raise StopIteration          # signal: I'm done
        value = self.current
        self.current -= 1
        return value


# Usage — works everywhere a for loop works:
for n in Countdown(5):
    print(n)
# 5 4 3 2 1 0

print(list(Countdown(3)))          # [3, 2, 1, 0]
print(sum(Countdown(10)))          # 55
print(max(Countdown(10)))          # 10

# Manual control:
cd = Countdown(2)
print(next(cd))    # 2
print(next(cd))    # 1
print(next(cd))    # 0
print(next(cd))    # StopIteration

5
4
3
2
1
0
[3, 2, 1, 0]
55
10
2
1
0


StopIteration: 

In [17]:
# vs Countdown (iterator IS the iterable):
cd = Countdown(3)
print(list(cd))    # [3, 2, 1, 0]
print(list(cd))    # []               ← exhausted, can't reuse!

[3, 2, 1, 0]
[]


**Example 4 — Infinite Iterator (real-world: ID generator)**

In [ ]:
class IDGenerator:
    """
    Infinite iterator — never raises StopIteration.
    Real use: generating unique IDs, sequence numbers, round-robin.
    """
    def __init__(self, prefix: str = "ID", start: int = 1):
        self.prefix  = prefix
        self.current = start

    def __iter__(self):
        return self

    def __next__(self):
        id_str       = f"{self.prefix}-{self.current:04d}"
        self.current += 1
        return id_str   # never raises StopIteration → infinite


gen = IDGenerator("ORD")
print(next(gen))    # "ORD-0001"
print(next(gen))    # "ORD-0002"
print(next(gen))    # "ORD-0003"

# Use itertools.islice to safely take N items from infinite iterators:
import itertools

first_five = list(itertools.islice(IDGenerator("USR"), 5))
print(first_five)
# ['USR-0001', 'USR-0002', 'USR-0003', 'USR-0004', 'USR-0005']

# ⚠️  NEVER do this — infinite loop:
# for id in IDGenerator():  ← runs forever, no StopIteration
#     print(id)

**Example 4 — File Chunker (real-world: processing large files)**

In [ ]:
class FileChunker:
    """
    Reads a large file in fixed-size chunks.
    Real use: processing CSVs, log files, uploads without loading all into RAM.
    """
    def __init__(self, filepath: str, chunk_size: int = 1024):
        self.filepath   = filepath
        self.chunk_size = chunk_size
        self._file      = None

    def __iter__(self):
        self._file = open(self.filepath, "r")
        return self

    def __next__(self):
        chunk = self._file.read(self.chunk_size)
        if not chunk:
            self._file.close()
            raise StopIteration
        return chunk

    # Resource safety — always close file even if loop breaks early
    def __del__(self):
        if self._file and not self._file.closed:
            self._file.close()


# Usage:
for chunk in FileChunker("large_log.txt", chunk_size=4096):
    process(chunk)   # never more than 4KB in memory at once

**Example 5 — Django QuerySet-Style Lazy Iterator**

In [ ]:
class LazyQueryIterator:
    """
    Mimics how Django QuerySets work — lazy evaluation,
    fetches from DB only when iteration starts.
    Real pattern: paginated DB reads, API cursor pagination.
    """
    def __init__(self, model_class, filters: dict, page_size: int = 100):
        self.model_class = model_class
        self.filters     = filters
        self.page_size   = page_size
        self._cache      = []
        self._offset     = 0
        self._exhausted  = False

    def __iter__(self):
        self._cache     = []
        self._offset    = 0
        self._exhausted = False
        return self

    def __next__(self):
        # Refill cache when empty
        if not self._cache:
            if self._exhausted:
                raise StopIteration

            # Simulate DB fetch (in real code: Model.objects.filter(**)..)
            page = self.model_class.objects.filter(**self.filters)[
                self._offset : self._offset + self.page_size
            ]
            results = list(page)

            if not results:
                raise StopIteration

            self._cache   = results
            self._offset += self.page_size

            if len(results) < self.page_size:
                self._exhausted = True   # last page — don't hit DB again

        return self._cache.pop(0)


# Usage — processes 1M rows with only 100 in memory at a time:
for user in LazyQueryIterator(User, {"is_active": True}, page_size=100):
    send_newsletter(user.email)

**The Iterator vs Generator — Know the Tradeoff**

In [ ]:
# Everything above, written as a generator function:
# Same behaviour, a fraction of the code.

def countdown(start: int):
    while start >= 0:
        yield start    # suspends here, returns value, resumes on next()
        start -= 1

def number_range(start, stop, step=1):
    current = start
    while current < stop:
        yield current
        current += step

def id_generator(prefix="ID", start=1):
    current = start
    while True:            # infinite — fine in a generator
        yield f"{prefix}-{current:04d}"
        current += 1


# Generators ARE iterators — same protocol, Python builds __next__ for you:
g = countdown(3)
print(type(g))              # <class 'generator'>
print(hasattr(g, "__next__"))  # True
print(hasattr(g, "__iter__"))  # True

print(next(g))   # 3
print(next(g))   # 2

```shell
When to write a CLASS iterator vs a generator:

CLASS iterator                    GENERATOR
──────────────────────────────    ──────────────────────────────
needs __del__ / resource cleanup  simple sequence of values
complex internal state            stateless or simple state
needs reusability (separate       one-shot iteration is fine
  iterable + iterator classes)
needs extra methods (reset,       just need __iter__/__next__
  peek, skip)
you want to expose a rich API     you want minimal code
```

---

**Complete Mental Model**
```shell
ITERABLE          has __iter__()          → can be used in for loop
                  returns an ITERATOR     → list, str, dict, your class

ITERATOR          has __iter__() + __next__()
                  __iter__ returns self
                  __next__ returns next value OR raises StopIteration
                  one-way, one-time

for x in obj:     calls iter(obj)  → obj.__iter__()
                  calls next(...)  → iterator.__next__()  on each step
                  catches StopIteration → breaks loop

Infinite iterator → never raises StopIteration
                    always use islice() or manual break

Class iterator    → full control, extra methods, resource management
Generator         → same protocol, Python builds __next__ from yield
```

---

> The interview answer: "An iterable has __iter__ and produces a fresh iterator on every call. An iterator has both __iter__ and __next__, tracks its own position, and is one-time use. The for loop is just a while True with iter() and next() and a StopIteration catch. I reach for a class-based iterator when I need resource cleanup, a reset mechanism, or complex internal state — otherwise a generator gives me the same protocol in a third of the code."